#Preliminary Analysis of Habitat Modification Scores 
####(main dataset is rss_tbl_river_habitat_survey - hms_class and subscores)

##River Habitat Survey HMS class

In [0]:
# Load packages
library(sparklyr) # connect to databricks
library(data.table) # data management
# GAMS - use mgcv for ordinal/catagorial data
#library(mgcv)
# Create a sparklyr connection.
sc <- spark_connect(method = "databricks")

##Load data
set parameters

In [0]:
# set parameters
param <- list(st="RSN")

# River habitat survey
# hms_class
rss_dt <- sdf_sql(sc, paste0("SELECT *
FROM prd_dash_lab.seda_restricted.rss_tbl_river_habitat_survey
WHERE network_id LIKE '", param$st, 
"%' AND completeness_band NOT LIKE '%Unacceptable%' ")) |> as.data.table()

In [0]:
# change to factors
rss_dt[, network_id := as.factor(network_id)]


In [0]:
#return only hms_class and hms_class_desc columns
rss_dt[, .(hms_class, hms_class_desc), keyby=hms_class] |> unique()

In [0]:
hms_class_table <- table(rss_dt[, hms_class])
# as %
hms_class_table  <- hms_class_table  / sum(hms_class_table) * 100
# Add names
classes <- c("Pristine/Near-natural", "Predominantly unmodified", "Obviously modified", "Significantly modified", "Severely modified")
names(hms_class_table) <- classes
# View table
hms_class_table

In [0]:
# plot overall percentages
par(bg="white", mar=c(5, 4, 4, 8))
cols <- c("#2ca25f", "#99d8c9", "#fec44f", "#fe9929", "#d95f0e")
# Plot
barplot(matrix(hms_class_table, ncol=1), col=cols,  ylim=c(0, 100), xlab="HMS Class")
# Legend
legend("right", inset=c(-0.45, 0), legend=names(hms_class_table),  fill=cols,  bty="n", xpd=TRUE)

In [0]:
# Alternative plot
par(bg="white")
barplot(matrix(hms_class_table, ncol=1), col=cols, beside=TRUE, ylim=c(0, 100), names.arg=names(hms_class_table), xlab="HMS Class", ylab="Proportion of Rivers (%)")

In [0]:
# Calculate overall % meeting high quality or above
hms_class_table[1] + hms_class_table[2]

###Calculate confidence intervals for the classes
*on the raw data

In [0]:
# Uses the binomial test to generate confidence intervals from the raw class data
# Thi gives class's proportion with a 95% CI
res <- rss_dt[, .N, by = hms_class][order(hms_class)]
res[, p := N / sum(N)]
res[, c("lo", "hi") := {
  ci <- sapply(N, \(n) binom.test(n, sum(N))$conf.int)
  .(ci[1, ], ci[2, ])
}]

# Remove row with na values for now.
res <- res[-6,]
res

In [0]:
# Alternative plot with confidence intervals
par(bg="white")
bar <- barplot(matrix(res[,p], ncol=1), col=cols, beside=TRUE, ylim=c(0, 1), names.arg=names(hms_class_table), xlab="HMS Class", ylab="Proportion of Rivers (%)")
# Add confidence intervals
arrows(x0=bar, y0=res[,lo], x1=bar, y1=res[,hi], code=3, angle=90, length=0.1)

##PCA - Identifying Habitat Subclasses Driving Modification Class

In [0]:
#list subclasses here as a vector
names(rss_dt)
hms_vars <- c("hms_reforced_bnk_bed_subscore", "hms_rsctned_bnk_subscore", "hms_berms_embnkments_subscore", "hms_culverts_subscore", "hms_wds_subscore", "hms_bridges_subscore", "hms_poaching_subscore", "hms_fords_subscore", "hms_ofall_dflector_subscore")

colSums(is.na(rss_dt[, ..hms_vars]))

#list the neworks that have missing data
rss_dt[is.na(hms_ofall_dflector_subscore),
       .(network_id, hms_class, survey_date)]

#remove NAs, create new data table
rss_pca <- rss_dt[complete.cases(rss_dt[, ..hms_vars])]

#PCA omitting the NAs
pca <- prcomp(rss_pca[, ..hms_vars], scale. = TRUE)


scores <- as.data.table(pca$x)
scores[, hms_class := rss_pca$hms_class]

summary(pca)

##Plot PCA

Scree plot - Variance explained by each principal component

In [0]:
#plot scree plot - shows how much variance each PC explains
plot(pca, type = "l")

Scatter plot - Principal component scores coloured by habitat modification class

In [0]:
scores <- as.data.table(pca$x)
scores[, hms_class := rss_pca$hms_class]

#plot PCA with arrows(subclasses)

scores[, hms_class_desc := factor(
  hms_class,
  levels = 1:5,
  labels = c(
   classes))]

cols <- c("darkgreen", "green", "yellow", "orange", "red")

#addiing arrows to pca plot
plot(scores$PC1, scores$PC2,
col = cols[scores$hms_class],pch = 16,xlab = "PC1 (19.0%)", ylab = "PC2 (16.0%)")

# Add arrows
arrows(0, 0, 
pca$rotation[,1] * 10,
pca$rotation[,2] * 8,
length = 0.1,col = "red")

# Add labels
text(
  pca$rotation[,1] * 10,
  pca$rotation[,2] * 8,
  labels = rownames(pca$rotation),
  col = "black",
  pos = 4,
  cex = 0.8)

legend(
  "bottomright",
  inset = c(-0.05, 0),
  legend = c(
    "Pristine/Near-natural",
    "Predominantly unmodified",
    "Obviously modified",
    "Significantly modified",
    "Severely modified"),
  col = cols, pch = 16, bty = "n", cex=0.8, xpd=TRUE)


Boxplot of PC1 by HMS Class - Distribution of PC1 scores across habitat modification classes

In [0]:
par(mar = c(10, 4, 4, 2))

boxplot(
  PC1 ~ hms_class_desc,
  data = scores,
  las = 2)

Contribution of HMS subclass scores to PCA axes

hms_reforced_bnk_bed_subscore, hms_wds_subscore and hms_bridges_subscore primarily associated with PC1 

hms_rsctned_bnk_subscore, hms_berms_embnkments_subscore and hms_poaching_subscore associated with PC2

In [0]:
pca$rotation

In [0]:
sort(abs(pca$rotation[,1]), decreasing = TRUE)


In [0]:
sort(abs(pca$rotation[,2]), decreasing = TRUE)

Correlation test- using both Spearman and Kendall methods (just as a test)

**Apply correction here BH or BY?**

In [0]:
cor.test(
  as.numeric(scores$hms_class),
  scores$PC1,
  method = "spearman"
)

In [0]:
cor.test(
  as.numeric(scores$hms_class),
  scores$PC1,
  method = "kendall")

In [0]:
names(scores)

Both Spearman and Kendall show a strong positive relationship association is strong (tau and rho) and highly significant (p < 0.001)

####Correlation for subclass variables here

In [0]:
#correlation test for subclass variables
cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_reforced_bnk_bed_subscore,
  method = "spearman")
#S = 314259067, p-value < 2.2e-16, rho=0.3039331 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$ hms_wds_subscore,
  method = "spearman")
#S = 336220504, p-value < 2.2e-16, rho=0.2552897 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_poaching_subscore, 
  method = "spearman")
#S = 502466099, p-value = 2.375e-05, rho=-0.1129353 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_fords_subscore,
  method = "spearman")
#S = 474355486, p-value = 0.05857, rho=-0.05067181 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_ofall_dflector_subscore,
  method = "spearman")
#S = 346398880, p-value < 2.2e-16, rho=0.2327452 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_wds_subscore ,
  method = "spearman")
#S = 336220504, p-value < 2.2e-16, rho=0.2552897 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_bridges_subscore,
  method = "spearman")
#S = 326397402, p-value < 2.2e-16, rho=0.2770474 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_culverts_subscore ,
  method = "spearman")
#S = 288945445, p-value < 2.2e-16, rho=0.3600014 

cor.test(
  as.numeric(rss_pca$hms_class),
  rss_pca$hms_reforced_bnk_bed_subscore,
  method = "spearman")
#S = 314259067, p-value < 2.2e-16, rho=0.3039331 

###More Variables added
####more environmental factors extracted from rss_tbl_samp_invertebrate 
##### !!re-do this bit and consider mutliple points for the same network id, maybe choose the values that are closest in date to when the HMS was recorded!!

In [0]:
#load data from prd_dash_lab.seda_restricted.rss_tbl_samp_invertebrate
#rss_invert_dt <- sdf_sql(sc, paste0("SELECT *
#FROM prd_dash_lab.seda_restricted.rss_tbl_samp_invertebrate
#WHERE network_id LIKE '", param$st, 
#"%' AND completeness_band NOT LIKE '%Unacceptable%' ")) |> as.data.table()

rss_invert_dt <- sdf_sql(sc, paste0("SELECT *
FROM prd_dash_lab.seda_restricted.rss_tbl_samp_invertebrate
WHERE network_id LIKE '", param$st, 
"%'")) |> as.data.table()

In [0]:
#choose variables from invert table to use in PCA
names(rss_invert_dt)
#primary_bank_use, primary_land_use -- too many catergories/too complex try again
#turbidity, shade, flow, detritus, overlay_silt_cover
invert_vars <- rss_invert_dt[, .(network_id, turbidity, shade, flow, detritus, overlay_silt_cover)]

for (v in names(invert_vars)) {
  cat("\n", v, "\n")
  print(sort(unique(invert_vars[[v]])))}


In [0]:
#convert them to ordinal in their natural order
invert_vars[, turbidity := factor(
  turbidity,
  levels = c("CLEAR: Water not visibly turbid", "SLIGHT: Visible turbidity but no significant effect on light penetration", "MODERATE: Significant effect on light penetration", "HIGH: Visibility limited to 10cm depth"),
  ordered = TRUE)]

invert_vars[, shade  := factor(
  shade,
  levels = c("NONE: Site is unshaded", "LIGHT: < 25%", "MODERATE: 25 - 50%", "HEAVY: > 50%"),
  ordered = TRUE)]  

invert_vars[, flow := factor(
  flow,
  levels = c("NO FLOW - Still water/unconnected pools", "DRY: No water", "LOW: Flow present but below normal level", "NORMAL: Normal flow rate", "HIGH: Faster than normal rates", "SPATE: Flood or near flood conditions, water course near bankfull"),
  ordered = TRUE)]

invert_vars[, detritus := factor(
  detritus,
  levels = c("NONE: None present", "LOCAL: Occasional patches - < 30% of area", "WIDESPREAD: 30 - 60% of area", "EXTENSIVE: > 61% of area."),
  ordered = TRUE)]

#invert_vars[, overlay_silt_cover := factor(
#  overlay_silt_cover,
#  levels = c("NONE: None present", "LOCAL: Occasional patches - < 30% of area", "WIDESPREAD: 30 - 60% of area", "EXTENSIVE: > 61% of area."),
#  ordered = TRUE)]


In [0]:
levels(invert_vars$turbidity)
levels(invert_vars$shade)
levels(invert_vars$flow)
levels(invert_vars$detritus)

In [0]:
hms_lookup <- unique(
  rss_dt[, .(network_id, hms_class, hms_class_desc)])

hms_lookup[, .N, by = network_id][N > 1]

In [0]:
There are some repeated networks - remove them

In [0]:
dup_ids <- rss_dt[, .N, by = network_id][N > 1, network_id]
#dup_ids
# RSN1807 RSN1129 RSN1132 RSN1262 RSN1946

hms_lookup <- rss_dt[
  !network_id %in% dup_ids,
  .(network_id, hms_class, hms_class_desc)]

hms_lookup[, .N, by = network_id][N > 1]

invert_vars <- merge(
  invert_vars,
  hms_lookup,
  by = "network_id",
  all.x = TRUE)
  names(invert_vars)

head(invert_vars)

In [0]:
env_vars <- c(
  "turbidity","shade","flow","detritus")

#covert to numeric
invert_num <- copy(invert_vars)

invert_num[, (env_vars) := lapply(.SD, as.numeric),
           .SDcols = env_vars]
#check
str(invert_num[, ..env_vars])

#check for NAs
colSums(is.na(invert_num[, ..env_vars]))

#remove NAs
pca_invert_data <- invert_num[complete.cases(invert_num[, ..env_vars])]

#PCA omitting the NAs
pca_invert <- prcomp(pca_invert_data[, ..env_vars],scale. = TRUE)

summary(pca_invert)

###Plot PCA

In [0]:
plot(pca_invert, type = "l")

In [0]:
scores_invert <- as.data.table(pca_invert$x)
scores_invert[, hms_class := pca_invert_data$hms_class]

#plot PCA with arrows(subclasses)

scores[, hms_class_desc := factor(
  hms_class,
  levels = 1:5,
  labels = c(
   classes))]

cols <- c("darkgreen", "green", "yellow", "orange", "red")

#addiing arrows to pca plot
plot(scores_invert$PC1, scores_invert$PC2,
col = cols[scores_invert$hms_class],pch = 16,xlab = "PC1 (31.9%)", ylab = "PC2 (28.5%)")

# Add arrows
arrows(0, 0, 
pca_invert$rotation[,1] * 5,
pca_invert$rotation[,2] * 5,
length = 0.1,col = "red")

# Add labels
text(
  pca_invert$rotation[,1] * 5,
  pca_invert$rotation[,2] * 5,
  labels = rownames(pca_invert$rotation),
  col = "black",
  pos = 4,
  cex = 0.8)

legend(
  "topleft",
  inset = c(0, 0),
  legend = c(
    "Pristine/Near-natural",
    "Predominantly unmodified",
    "Obviously modified",
    "Significantly modified",
    "Severely modified"),
  col = cols, pch = 16, bty = "n", cex=0.8, xpd=TRUE)


Does not seem like there is a strong associationi between the PCA and HMS class

In [0]:
sort(abs(pca_invert$rotation[,1]), decreasing = TRUE)


In [0]:
sort(abs(pca_invert$rotation[,2]), decreasing = TRUE)

In [0]:
cor.test(
  as.numeric(scores_invert$hms_class),
  scores_invert$PC1,
  method = "kendall")

Although it is significant tau<-0.1 is negligible, and no meaningful relationship :(  PC2 resulted in similar values p<0.001 tau<-0.1

Spearman or Kendall test?

In [0]:
cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$turbidity,
  method = "spearman")

cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$shade,
  method = "spearman")

cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$flow,
  method = "spearman")

cor.test(
  as.numeric(pca_invert_data$hms_class),
  pca_invert_data$detritus,
  method = "spearman")

The correlations above resulted in significant p values but rho <0.1 - therefore negligible/weak relationship


Trying primary_land_use and primary_bank_use, unsure if this has worked. Some observations have 2 or more values, Eg simple... and complex...is the second one supposed to be secondary bank use?

In [0]:
use_vars <- rss_invert_dt[, .(network_id, primary_land_use, primary_bank_use)]

In [0]:
use_vars_1 <- merge(
  use_vars,
  hms_lookup,
  by = "network_id",
  all.x = TRUE)
  names(use_vars_1)

In [0]:
chisq.test(
  table(
    use_vars_1$primary_bank_use,
    use_vars_1$hms_class))

In [0]:
tab <- prop.table(
  table(use_vars_1$primary_bank_use,
        use_vars_1$hms_class), margin = 1)

round(tab, 2)